# SCT Per-Component LR Experiment
**Make-or-break test:** Can per-component learning rates close the quality gap?

- Attention + norms + embeddings: LR = 2e-5 (dense baseline rate)
- SCT MLP factors (U, s, V): LR = 5e-4 (25x higher for spectral recovery)

SmolLM2-1.7B, Alpaca, rank 32, 2000 steps, T4 GPU.

**Success criterion:** PPL drops from ~65-87 (uniform LR) toward single digits (~3.6 dense baseline).

In [ ]:
# Cell 1: Setup
!pip install -q transformers datasets
import torch, math, time, json
import torch.nn as nn

assert torch.cuda.is_available(), "No GPU. Go to Runtime > Change runtime type > T4"
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f"GPU: {gpu} ({vram:.1f} GB)")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# Cell 2: SpectralLinear

def safe_qr(M):
    """QR with sign correction. CPU path avoids T4 driver edge cases."""
    dev = M.device
    Q, R = torch.linalg.qr(M.float().cpu())
    return (Q * torch.sign(torch.diag(R))).to(dev).to(M.dtype)

class SpectralLinear(nn.Module):
    """W = U diag(s) V^T with Stiefel QR retraction."""
    def __init__(self, in_features, out_features, rank=32):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.rank = min(rank, min(in_features, out_features))
        U = torch.randn(in_features, self.rank) / math.sqrt(in_features)
        V = torch.randn(out_features, self.rank) / math.sqrt(out_features)
        Q_U, R_U = torch.linalg.qr(U)
        Q_V, R_V = torch.linalg.qr(V)
        self.U = nn.Parameter((Q_U * torch.sign(torch.diag(R_U)))[:, :self.rank])
        self.V = nn.Parameter((Q_V * torch.sign(torch.diag(R_V)))[:, :self.rank])
        self.s = nn.Parameter(torch.ones(self.rank))

    def forward(self, x):
        # Cast factors to input dtype for mixed precision
        U = self.U.to(x.dtype)
        s = self.s.to(x.dtype)
        V = self.V.to(x.dtype)
        return (x @ U) * s @ V.T

    @torch.no_grad()
    def retract(self):
        self.U.data = safe_qr(self.U.data)[:, :self.rank]
        self.V.data = safe_qr(self.V.data)[:, :self.rank]

def retract_all(model):
    for m in model.modules():
        if isinstance(m, SpectralLinear):
            m.retract()

print("SpectralLinear ready.")

In [ ]:
# Cell 3: Load model and convert MLP layers
from transformers import AutoModelForCausalLM, AutoTokenizer

RANK = 32
DEVICE = "cuda"

print("Loading SmolLM2-1.7B...")
model = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceTB/SmolLM2-1.7B",
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# Step 1: Freeze everything
for p in model.parameters():
    p.requires_grad = False

# Step 2: Convert MLP layers to SpectralLinear (creates new trainable params)
converted = 0
for name, module in model.named_modules():
    for attr in ['gate_proj', 'up_proj', 'down_proj']:
        if hasattr(module, attr):
            linear = getattr(module, attr)
            if isinstance(linear, nn.Linear):
                spec = SpectralLinear(linear.in_features, linear.out_features, RANK)
                spec = spec.to(linear.weight.device)
                with torch.no_grad():
                    W = linear.weight.data.float()
                    U_full, S_full, Vt_full = torch.linalg.svd(W, full_matrices=False)
                    k = spec.rank
                    spec.U.data = Vt_full[:k, :].T.contiguous()
                    spec.V.data = U_full[:, :k].contiguous()
                    spec.s.data = S_full[:k].contiguous()
                setattr(module, attr, spec)
                del linear
                converted += 1
print(f"Converted {converted} MLP layers to SpectralLinear (rank={RANK})")

# Step 3: CRITICAL FIX - Unfreeze attention projections and norms
# This is the bug fix: without this, attention stays frozen and
# the per-component LR experiment is meaningless
unfrozen_attn = 0
unfrozen_norm = 0
for name, param in model.named_parameters():
    if any(k in name for k in ["q_proj", "k_proj", "v_proj", "o_proj"]):
        param.requires_grad = True
        unfrozen_attn += param.numel()
    elif any(k in name for k in ["layernorm", "input_layernorm", "post_attention_layernorm", "norm"]):
        param.requires_grad = True
        unfrozen_norm += param.numel()

print(f"Unfrozen attention params: {unfrozen_attn:,}")
print(f"Unfrozen norm params: {unfrozen_norm:,}")

# Note: embeddings stay frozen to save memory on T4
# They account for ~100M params and freezing them is standard practice

torch.cuda.empty_cache()
mem_after_load = torch.cuda.max_memory_allocated() / 1e9
print(f"\nGPU memory after load: {mem_after_load:.1f} GB")

In [ ]:
# Cell 4: Verify param groups and create optimizers

LOW_LR = 2e-5    # Dense baseline rate for attention + norms
HIGH_LR = 5e-4   # 25x higher for SCT spectral factors

dense_params = []   # attention projections + norms
sct_params = []     # SpectralLinear U, s, V

for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if any(k in name for k in ["q_proj", "k_proj", "v_proj", "o_proj",
                                "layernorm", "input_layernorm",
                                "post_attention_layernorm", "norm"]):
        dense_params.append(param)
    else:
        # These are the SpectralLinear .U, .s, .V params
        sct_params.append(param)

dense_count = sum(p.numel() for p in dense_params)
sct_count = sum(p.numel() for p in sct_params)
total_trainable = dense_count + sct_count

print(f"Dense params (attn+norm):  {dense_count:>12,}  LR = {LOW_LR}")
print(f"SCT params (U, s, V):      {sct_count:>12,}  LR = {HIGH_LR}")
print(f"Total trainable:           {total_trainable:>12,}")
print(f"LR ratio:                  {HIGH_LR/LOW_LR:.0f}x")

# Sanity check
assert len(dense_params) > 0, "BUG: No dense params found. Attention is still frozen."
assert len(sct_params) > 0, "BUG: No SCT params found. Conversion failed."
assert dense_count > 100_000_000, f"BUG: Only {dense_count} dense params. Expected ~400M."

optimizer = torch.optim.AdamW([
    {"params": dense_params, "lr": LOW_LR},
    {"params": sct_params, "lr": HIGH_LR},
], weight_decay=0.01)

print(f"\nSingle AdamW optimizer with 2 param groups. Ready.")

In [ ]:
# Cell 5: Load and tokenize data
from datasets import load_dataset

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-1.7B")
tokenizer.pad_token = tokenizer.eos_token

ds = load_dataset("tatsu-lab/alpaca", split="train")

MAX_SEQ_LEN = 256  # Shorter to fit T4 memory

def format_example(ex):
    if ex.get("input", "").strip():
        return f"### Instruction:\n{ex['instruction']}\n\n### Input:\n{ex['input']}\n\n### Response:\n{ex['output']}"
    return f"### Instruction:\n{ex['instruction']}\n\n### Response:\n{ex['output']}"

texts = [format_example(ex) for ex in ds]
encodings = tokenizer(texts, truncation=True, max_length=MAX_SEQ_LEN,
                      padding="max_length", return_tensors="pt")

print(f"Data: {len(texts)} examples, seq_len={MAX_SEQ_LEN}")

In [ ]:
# Cell 6: Training loop

STEPS = 2000
BATCH_SIZE = 2
LOG_EVERY = 100
DEVICE = "cuda"

print(f"{'='*65}")
print(f"  SCT Per-Component LR Experiment")
print(f"  Dense params (attn+norm): LR = {LOW_LR}")
print(f"  SCT params (U, s, V):     LR = {HIGH_LR}")
print(f"  Steps: {STEPS}  Batch: {BATCH_SIZE}  Seq: {MAX_SEQ_LEN}  Rank: {RANK}")
print(f"{'='*65}\n")

model.train()
input_ids = encodings["input_ids"]
attention_mask = encodings["attention_mask"]
n_samples = input_ids.shape[0]

losses = []
step_times = []
t_start = time.time()

for step in range(1, STEPS + 1):
    idx = torch.randint(0, n_samples, (BATCH_SIZE,))
    xb = input_ids[idx].to(DEVICE)
    mb = attention_mask[idx].to(DEVICE)
    labels = xb.clone()
    labels[mb == 0] = -100

    t0 = time.time()

    optimizer.zero_grad()
    outputs = model(input_ids=xb, attention_mask=mb, labels=labels)
    loss = outputs.loss
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    retract_all(model)

    step_time = time.time() - t0
    loss_val = loss.item()
    losses.append({"step": step, "loss": loss_val, "time": step_time})
    step_times.append(step_time)

    if step % LOG_EVERY == 0 or step == 1:
        ppl = math.exp(min(loss_val, 20))
        avg_loss = sum(l["loss"] for l in losses[-LOG_EVERY:]) / min(LOG_EVERY, len(losses))
        avg_time = sum(step_times[-LOG_EVERY:]) / len(step_times[-LOG_EVERY:])
        eta = (STEPS - step) * avg_time / 60
        mem = torch.cuda.max_memory_allocated() / 1e9
        print(f"  Step {step:>5d}/{STEPS}  Loss: {loss_val:.4f}  "
              f"Avg: {avg_loss:.4f}  PPL: {ppl:.1f}  "
              f"Step: {avg_time:.2f}s  GPU: {mem:.1f}GB  ETA: {eta:.1f}min")

total_time = time.time() - t_start
print(f"\nTraining complete in {total_time/60:.1f} minutes.")

In [ ]:
# Cell 7: Results and comparison
import matplotlib.pyplot as plt

final_loss = losses[-1]["loss"]
final_ppl = math.exp(min(final_loss, 20))
avg_final = sum(l["loss"] for l in losses[-50:]) / 50
avg_final_ppl = math.exp(min(avg_final, 20))
peak_mem = torch.cuda.max_memory_allocated() / 1e9

print(f"{'='*65}")
print(f"  RESULTS: Per-Component LR Experiment")
print(f"{'='*65}")
print(f"  Final loss (last step):    {final_loss:.4f}")
print(f"  Final PPL (last step):     {final_ppl:.1f}")
print(f"  Avg loss (last 50 steps):  {avg_final:.4f}")
print(f"  Avg PPL (last 50 steps):   {avg_final_ppl:.1f}")
print(f"  Peak GPU memory:           {peak_mem:.1f} GB")
print(f"  Total time:                {total_time/60:.1f} min")
print(f"{'='*65}")
print(f"")
print(f"  COMPARISON:")
print(f"  Dense baseline PPL:        3.6")
print(f"  SCT uniform LR PPL:        ~65-87")
print(f"  SCT per-component LR PPL:  {avg_final_ppl:.1f}  <-- THIS RUN")
print(f"")
if avg_final_ppl < 10:
    print(f"  STATUS: GAP CLOSED. SCT is viable.")
elif avg_final_ppl < 30:
    print(f"  STATUS: PARTIAL IMPROVEMENT. LR helps but gap remains.")
else:
    print(f"  STATUS: NO IMPROVEMENT. Problem is deeper than LR.")
print(f"{'='*65}")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

steps_list = [l["step"] for l in losses]
loss_list = [l["loss"] for l in losses]

# Smoothed
w = 30
smooth = [sum(loss_list[max(0,i-w+1):i+1]) / (i - max(0,i-w+1) + 1) for i in range(len(loss_list))]

ax1.plot(steps_list, loss_list, alpha=0.15, color='#FF5722')
ax1.plot(steps_list, smooth, color='#FF5722', linewidth=2, label=f'SCT per-comp LR (r={RANK})')
ax1.axhline(y=1.29, color='#2196F3', linestyle='--', alpha=0.7, label='Dense baseline (1.29)')
ax1.set_xlabel('Step')
ax1.set_ylabel('Loss')
ax1.set_title('Per-Component LR: Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ppl_smooth = [math.exp(min(l, 20)) for l in smooth]
ax2.plot(steps_list, ppl_smooth, color='#FF5722', linewidth=2, label=f'SCT per-comp LR (r={RANK})')
ax2.axhline(y=3.6, color='#2196F3', linestyle='--', alpha=0.7, label='Dense baseline (3.6)')
ax2.set_xlabel('Step')
ax2.set_ylabel('Perplexity')
ax2.set_title('Per-Component LR: Perplexity')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_yscale('log')

fig.text(0.5, 0.01,
         f'SmolLM2-1.7B | Alpaca | {torch.cuda.get_device_name(0)} | '
         f'Attn LR: {LOW_LR} | SCT LR: {HIGH_LR} | Rank {RANK} | EctoSpace/SCT',
         ha='center', fontsize=9, color='gray')

plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('sct_per_component_lr.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved: sct_per_component_lr.png")

In [ ]:
# Cell 8: Save and download

summary = {
    "experiment": "SCT per-component LR",
    "model": "SmolLM2-1.7B",
    "dataset": "alpaca",
    "hardware": torch.cuda.get_device_name(0),
    "rank": RANK,
    "dense_lr": LOW_LR,
    "sct_lr": HIGH_LR,
    "steps": STEPS,
    "batch_size": BATCH_SIZE,
    "seq_len": MAX_SEQ_LEN,
    "dense_trainable_params": dense_count,
    "sct_trainable_params": sct_count,
    "total_trainable_params": total_trainable,
    "final_loss": final_loss,
    "final_ppl": final_ppl,
    "avg_final_loss_50": avg_final,
    "avg_final_ppl_50": avg_final_ppl,
    "peak_gpu_gb": peak_mem,
    "total_time_min": total_time / 60,
    "comparison": {
        "dense_ppl": 3.6,
        "sct_uniform_lr_ppl_range": "65-87",
        "sct_per_component_lr_ppl": avg_final_ppl
    }
}

with open("sct_per_component_lr_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

with open("sct_per_component_lr_losses.json", "w") as f:
    json.dump(losses, f)

print("Results saved.")

from google.colab import files
files.download('sct_per_component_lr.png')
files.download('sct_per_component_lr_summary.json')
files.download('sct_per_component_lr_losses.json')
print("All files downloaded.")